In [2]:
import pandas as pd
from pathlib import Path
import os
import sys

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import *

import numpy as np

import plotly.graph_objects as go

pd.set_option('display.max_columns', None)

In [3]:
# Criação da sessão Spark local
spark = (
    SparkSession
    .builder
    .config("spark.driver.memory", "4g") 
    .config("spark.executor.memory", "4g") 
    .config("spark.python.worker.faulthandler.enabled", "true") # se algum worker crashar de novo, imprime o traceback nativo real
    .master("local[*]")
    .appName("eda_cycle_start_gk")
    .getOrCreate()
)

## Funções

### Agregações de Features x Target

In [4]:
AGG_FUNCS_FEATURES = {
    "avg": F.mean,
    "median": F.median,
    "min": F.min,
    "max": F.max,
    #"var": F.variance,
    "p95": lambda c: F.percentile_approx(c, 0.95),
}

AGG_FUNCS_TARGET = {
    "avg": F.mean,
    "sum": F.sum,
    "max": F.max,
}


def build_feature_target_stats_df(df, features, target, group_cols, consistency_flag="is_tracking_inconsistent"):
    """
    Agrega feature(s) e o target por group_cols num único groupBy().agg().
    As features usam F.when(~consistency_flag, coluna) — o próprio agregador
    (mean/median/min/max/variance/percentile_approx) ignora os NULLs gerados
    pelo when, então eventos com tracking inconsistente simplesmente não
    contribuem pra estatística da feature, sem precisar filtrar o df antes
    (surface_area, stretch_index etc. ficam distorcidas por esses eventos —
    ver sanity_check_features_tracking.ipynb). O target usa o MESMO
    F.when(~consistency_flag, coluna) — sem isso, o evento de pico/mínimo
    do target poderia cair justamente num evento com tracking inconsistente
    (excluído do cálculo das features), descasando qual evento "conta" pro
    target do que "conta" pras features dentro do mesmo ciclo.

    Parâmetros
    ----------
    df : DataFrame Spark contendo as colunas de features/target, group_cols
         e a coluna consistency_flag.
    features : str ou list[str]
        Nome(s) da(s) coluna(s) de feature a agregar.
    target : str
        Nome da coluna alvo (recebe as agregações de AGG_FUNCS_TARGET).
    group_cols : list[str]
        Colunas de agrupamento (ex: chave de ciclo de posse).
    consistency_flag : str
        Coluna booleana que marca tracking inconsistente (True = inconsistente).

    Retorna
    -------
    df_agg : DataFrame Spark, uma linha por grupo, com as colunas agregadas
             (formato "{agregacao}_{coluna}", ex: "avg_surface_area").
    """
    if isinstance(features, str):
        features = [features]

    agg_exprs = []

    for col_name in features:
        consistent_col = F.when(~F.col(consistency_flag), F.col(col_name))
        for prefix, func in AGG_FUNCS_FEATURES.items():
            alias = f"{prefix}_{col_name}"
            agg_exprs.append(F.round(func(consistent_col), 3).alias(alias))

    consistent_target = F.when(~F.col(consistency_flag), F.col(target))
    for prefix, func in AGG_FUNCS_TARGET.items():
        alias = f"{prefix}_{target}"
        agg_exprs.append(F.round(func(consistent_target), 3).alias(alias))

    df_agg = df.groupBy(*group_cols).agg(*agg_exprs)

    return df_agg

In [5]:
def plot_correlation_heatmap(df_agg, features, target, target_aggs=None, title_suffix=""):
    """
    Monta UM heatmap de correlação (Spearman) cruzando TODAS as agregações
    de TODAS as features pedidas (AGG_FUNCS_FEATURES — avg/median/min/max/
    p95, uma coluna por combinação agregação×feature) com a(s) agregação(ões)
    do target pedida(s) em target_aggs. Aceita uma feature (str) ou várias
    (list) — a chamada é sempre a mesma, só muda quantas colunas entram na
    matriz.

    Parâmetros
    ----------
    df_agg : DataFrame Spark ou pandas retornado por build_feature_target_stats_df.
    features : str ou list[str]
        Nome(s) da(s) feature(s) originais (sem prefixo de agregação).
    target : str
        Nome da coluna alvo original (sem prefixo de agregação).
    target_aggs : str ou list[str]
        Chave(s) de AGG_FUNCS_TARGET a usar pro target (default: todas).
    title_suffix : str
        Texto extra concatenado ao título do heatmap (ex: nome do time,
        quando plotado via plot_correlation_heatmap_by_team).
    """
    if isinstance(features, str):
        features = [features]
    if target_aggs is None:
        target_aggs = list(AGG_FUNCS_TARGET.keys())
    elif isinstance(target_aggs, str):
        target_aggs = [target_aggs]

    df_agg_pd = df_agg.toPandas() if not isinstance(df_agg, pd.DataFrame) else df_agg

    agg_names = list(AGG_FUNCS_FEATURES.keys())
    feature_cols = [f"{prefix}_{feature}" for feature in features for prefix in agg_names]
    target_cols = [f"{ta}_{target}" for ta in target_aggs]

    corr = df_agg_pd[feature_cols + target_cols].corr(method='spearman')

    n = len(corr.columns)

    fig = go.Figure(
        data=go.Heatmap(
            z=corr.values,
            x=corr.columns,
            y=corr.columns,
            colorscale='RdBu',
            zmin=-1,
            zmax=1,
            text=corr.round(3).values,
            texttemplate="%{text}",
            colorbar=dict(title="Correlação"),
        )
    )

    fig.update_layout(
        title=f"Correlação (Spearman) — agregações das features x {'/'.join(target_aggs)}_{target}{title_suffix}",
        template="simple_white",
        height=700,
        width=1000,
        xaxis=dict(tickangle=-45),
    )

    fig.show()

In [6]:
def plot_correlation_heatmap_by_team(df_agg, features, target, team_col, feature_agg, target_aggs='max', max_teams=5):
    """
    Um heatmap por time — diferente de plot_correlation_heatmap (que expande
    TODAS as agregações de AGG_FUNCS_FEATURES por feature): aqui cada
    feature entra com UMA única agregação (`feature_agg`, ex: "avg"), a
    mesma pra todas as features, e o target entra com a(s) agregação(ões)
    de `target_aggs` (default "max"). Serve pra ver se a correlação entre
    as features (nessa agregação) e o target se sustenta time a time, ou se
    é puxada por poucos times específicos.

    Parâmetros
    ----------
    df_agg : DataFrame Spark ou pandas retornado por build_feature_target_stats_df,
        contendo a coluna team_col (ex: "defendingTeamName") entre as chaves
        de agrupamento.
    features : str ou list[str]
        Nome(s) da(s) feature(s) originais (sem prefixo de agregação).
    target : str
        Nome da coluna alvo original (sem prefixo de agregação).
    team_col : str
        Nome da coluna de time usada como chave de agrupamento extra.
    feature_agg : str
        Chave de AGG_FUNCS_FEATURES a usar pra TODAS as features (ex: "avg").
    target_aggs : str ou list[str]
        Chave(s) de AGG_FUNCS_TARGET a usar pro target (default "max").
    max_teams : int
        Quantidade máxima de times (heatmaps) a plotar, pra não gerar uma
        lista enorme de gráficos (default 5).
    """
    if isinstance(features, str):
        features = [features]
    if isinstance(target_aggs, str):
        target_aggs = [target_aggs]

    df_agg_pd = df_agg.toPandas() if not isinstance(df_agg, pd.DataFrame) else df_agg

    feature_cols = [f"{feature_agg}_{feature}" for feature in features]
    target_cols = [f"{ta}_{target}" for ta in target_aggs]

    teams = sorted(df_agg_pd[team_col].dropna().unique())[:max_teams]

    for team in teams:
        df_team_pd = df_agg_pd[df_agg_pd[team_col] == team]
        corr = df_team_pd[feature_cols + target_cols].corr(method='spearman')

        fig = go.Figure(
            data=go.Heatmap(
                z=corr.values,
                x=corr.columns,
                y=corr.columns,
                colorscale='RdBu',
                zmin=-1,
                zmax=1,
                text=corr.round(3).values,
                texttemplate="%{text}",
                colorbar=dict(title="Correlação"),
            )
        )

        fig.update_layout(
            title=f"Correlação (Spearman) — features ({feature_agg}) x {'/'.join(target_aggs)}_{target} — {team}",
            template="simple_white",
            height=700,
            width=1000,
            xaxis=dict(tickangle=-45),
        )

        fig.show()

In [7]:
def _plot_position_bin_heatmap(df_agg_pd, features, target, feature_agg, target_agg, bin_col, title_suffix=""):
    """Monta a matriz features x faixas e desenha UM heatmap — usada por plot_correlation_by_position_bin e plot_correlation_by_position_bin_by_team."""
    target_col = f"{target_agg}_{target}"
    bin_values = [b for b in df_agg_pd[bin_col].cat.categories if b in df_agg_pd[bin_col].values]

    z = []
    for feature in features:
        feature_col = f"{feature_agg}_{feature}"
        row = []
        for b in bin_values:
            df_bin = df_agg_pd[df_agg_pd[bin_col] == b]
            corr_value = (
                df_bin[[feature_col, target_col]].corr(method='spearman').iloc[0, 1]
                if len(df_bin) >= 3 else np.nan
            )
            row.append(corr_value)
        z.append(row)

    x_labels = [str(b) for b in bin_values]
    z = np.array(z)

    fig = go.Figure(
        data=go.Heatmap(
            z=z,
            x=x_labels,
            y=features,
            colorscale='RdBu',
            zmin=-1,
            zmax=1,
            text=np.round(z, 3),
            texttemplate="%{text}",
            colorbar=dict(title="Correlação"),
        )
    )

    fig.update_layout(
        title=f"Correlação (Spearman) por faixa de {bin_col} — features ({feature_agg}) x {target_agg}_{target}{title_suffix}<br><sup>Faixas: {' | '.join(x_labels)}</sup>",
        template="simple_white",
        height=700,
        width=1000,
        xaxis=dict(tickangle=-45),
    )

    fig.show()


def plot_correlation_by_position_bin(df_agg_pd, features, target, feature_agg, target_agg, bin_col):
    """
    Um único heatmap: linhas = features, colunas = faixas de bin_col,
    células = correlação (Spearman) daquela feature com o target, calculada
    SEPARADAMENTE dentro de cada faixa (filtra df_agg_pd pela faixa antes de
    rodar .corr()). Serve pra checar se a correlação feature x target se
    sustenta em diferentes zonas do campo, ou se é puxada por confundimento
    posicional — ameaça e compactação defensiva tendem a subir juntas perto
    do próprio gol, então uma correlação alta na base inteira pode ser só
    "as duas coisas acontecem perto do gol", não um efeito real da forma
    defensiva. Ao contrário de plot_correlation_heatmap, aqui cada feature
    entra com UMA única agregação (não expande AGG_FUNCS_FEATURES inteiro),
    já que o objetivo é comparar zonas, não comparar tipos de agregação.

    Parâmetros
    ----------
    df_agg_pd : pandas.DataFrame
        Já deve conter as colunas "{agg}_{feature}"/"{target_agg}_{target}"
        e a coluna bin_col (categórica, ex: gerada por pd.cut).
    features : str ou list[str]
        Nome(s) da(s) feature(s) originais (sem prefixo de agregação).
    target : str
        Nome da coluna alvo original (sem prefixo de agregação).
    feature_agg : str
        Chave de AGG_FUNCS_FEATURES a usar — a MESMA agregação pra todas as
        features (ex: "min").
    target_agg : str
        Chave de AGG_FUNCS_TARGET a usar pro target (uma só, ex: "max").
    bin_col : str
        Nome da coluna categórica de faixas de posição (ex: "ball_x_at_max_bin").
    """
    if isinstance(features, str):
        features = [features]

    _plot_position_bin_heatmap(df_agg_pd, features, target, feature_agg, target_agg, bin_col)


def plot_correlation_by_position_bin_by_team(df_agg_pd, features, target, team_col, feature_agg, target_agg, bin_col, max_teams=5):
    """
    Igual a plot_correlation_by_position_bin, mas um heatmap por time —
    filtra df_agg_pd pelos valores distintos de team_col (limitado a
    max_teams, pra não gerar uma quantidade enorme de heatmaps) e roda o
    mesmo cálculo por faixa dentro de cada time, com o nome do time no
    título. Serve pra ver se o padrão por zona de campo (visto na base
    inteira) se sustenta time a time.

    Parâmetros
    ----------
    df_agg_pd : pandas.DataFrame
        Já deve conter as colunas "{agg}_{feature}"/"{target_agg}_{target}",
        a coluna bin_col e a coluna team_col.
    features : str ou list[str]
        Nome(s) da(s) feature(s) originais (sem prefixo de agregação).
    target : str
        Nome da coluna alvo original (sem prefixo de agregação).
    team_col : str
        Nome da coluna de time usada como chave de agrupamento extra.
    feature_agg : str
        Chave de AGG_FUNCS_FEATURES a usar — a MESMA agregação pra todas as
        features (ex: "min").
    target_agg : str
        Chave de AGG_FUNCS_TARGET a usar pro target (uma só, ex: "max").
    bin_col : str
        Nome da coluna categórica de faixas de posição (ex: "ball_x_at_max_bin").
    max_teams : int
        Quantidade máxima de times (heatmaps) a plotar (default 5).
    """
    if isinstance(features, str):
        features = [features]

    teams = sorted(df_agg_pd[team_col].dropna().unique())[:max_teams]

    for team in teams:
        df_team_pd = df_agg_pd[df_agg_pd[team_col] == team]
        _plot_position_bin_heatmap(
            df_team_pd, features, target, feature_agg, target_agg, bin_col, title_suffix=f" — {team}"
        )

In [8]:
# caminho pra pasta com dados
data_folder_path = Path().resolve().parent.parent / "data"

In [9]:
# base de ameaça criada
threat_dataset_path = str(data_folder_path / "threat_dataset")
features_dataset_path = str(data_folder_path / "features_dataset")

df_threat = spark.read.parquet(threat_dataset_path)
df_features = spark.read.csv(features_dataset_path, header=True, inferSchema=True)

In [10]:
threat_cols = [
    'gameId',
    'competitionId',
    'season',
    'date',
    'eventId',
    'period',
    'startGameClock',
    'startFormattedGameClock',
    'homeTeam',
    'flipped_homeTeam',
    'eventType',
    'eventTypeDescription',
    'eventSubTypeDescription',
    'eventOutcomeDescription',
    'eventPlayerName',
    'eventPlayerPositionType',
    'eventPlayerPositionGroup',
    'eventTeamName',
    'competitionName',
    'homeTeamName',
    'opponentTeamName',
    'possession_id',
    'attackers',
    'defenders',
    'attackingPlayersNorm',
    'defendingPlayersNorm',
    'ballsNorm',
    'threat_score',
    'threat_score_impact'
]

In [11]:
df_threat_features = (
    df_threat.select(*threat_cols)
    .join(
        df_features,
        on=["competitionId", "season", "gameId", "eventId"],
        how='inner'
    )
)

# Flag auxiliar: indica se esse evento é o primeiro da sua posse
w_pos = Window.partitionBy("competitionId", "season", "gameId").orderBy("startGameClock")

df_threat_features = df_threat_features.withColumn(
    "first_cycle_event",
    F.lag("possession_id", 1).over(w_pos).isNull() |
    (F.lag("possession_id", 1).over(w_pos) != F.col("possession_id"))
)

# Time que estava DEFENDENDO no ciclo/evento (oposto de quem a flag
# homeTeam identifica como atacante) — é o time que as features de forma
# (surface_area, stretch_index etc.) descrevem, então é a chave natural
# pra quebrar as correlações por time.
df_threat_features = df_threat_features.withColumn(
    "defendingTeamName",
    F.when(F.col("homeTeam"), F.col("opponentTeamName")).otherwise(F.col("homeTeamName"))
)

# Posição x da bola no evento (mesma convenção de target_engineering.ipynb:
# ballsNorm é array de 1 elemento {x, y, z, visibility}) — usada mais
# adiante pra estratificar a correlação por zona de campo.
df_threat_features = df_threat_features.withColumn(
    "ball_x",
    F.get("ballsNorm", 0)["x"]
)

df_threat_features.show()

+-------------+---------+------+--------------------+----------+------+--------------+-----------------------+--------+----------------+------------+--------------------+-----------------------+-----------------------+------------------+-----------------------+------------------------+--------------+---------------+--------------+----------------+-------------+---------+---------+--------------------+--------------------+--------------------+------------+-------------------+------------------------+------------+-------------+-----------+----------+-------------+------------------+--------------------------+-------------------------+------------+------------+------------+-----------------------+-----------------------+-----------------+-----------------+------+
|competitionId|   season|gameId|             eventId|      date|period|startGameClock|startFormattedGameClock|homeTeam|flipped_homeTeam|   eventType|eventTypeDescription|eventSubTypeDescription|eventOutcomeDescription|   eventPla

## Abordagem 1: Ciclos de posse começando no goleiro e com todos os 11 jogadores do time defendendo atrás da linha da bola

In [12]:
df_cycle_start_gk_keys = (
    df_threat_features
    .filter(
        (F.col('first_cycle_event')) &
        (F.col('eventPlayerPositionType') == 'GK') &
        (F.col('defenders') == 11)
    )
    .select('competitionId', 'season', 'gameId', 'possession_id')
)

df_cycle_start_gk = (
    df_threat_features
    .join(
        F.broadcast(df_cycle_start_gk_keys),
        on=['competitionId', 'season', 'gameId', 'possession_id'],
        how='inner'
    )
)
df_cycle_start_gk = df_cycle_start_gk.cache()
df_cycle_start_gk.show(5)

+-------------+---------+------+-------------+--------------------+----------+------+--------------+-----------------------+--------+----------------+---------+--------------------+-----------------------+-----------------------+----------------+-----------------------+------------------------+-------------+---------------+------------+----------------+---------+---------+--------------------+--------------------+--------------------+------------+-------------------+------------------------+------------+-------------+-----------+----------+-------------+------------------+--------------------------+-------------------------+------------+------------+------------+-----------------------+-----------------------+-----------------+-----------------+------+
|competitionId|   season|gameId|possession_id|             eventId|      date|period|startGameClock|startFormattedGameClock|homeTeam|flipped_homeTeam|eventType|eventTypeDescription|eventSubTypeDescription|eventOutcomeDescription| eventPl

In [13]:
qtd_eventos = df_cycle_start_gk.count()
qtd_eventos_total = df_threat_features.count()

print(f'Quantidade de eventos: {qtd_eventos}')
print(f'Quantidade de eventos total na temporada: {qtd_eventos_total}')
print(f'Quantidade de eventos em relação ao total: {qtd_eventos / qtd_eventos_total:.2%}')

Quantidade de eventos: 27809
Quantidade de eventos total na temporada: 448193
Quantidade de eventos em relação ao total: 6.20%


In [14]:
df_possession_agg = (
    df_cycle_start_gk
    .groupBy('competitionId', 'season', 'gameId', 'possession_id')
    .agg(F.countDistinct(F.col('eventId')).alias('qtd_eventos')
    )
    .sort('qtd_eventos', ascending=False)
)

qtd_ciclos = df_possession_agg.count()
qtd_ciclos_total = df_threat_features.select('competitionId', 'season', 'gameId', 'possession_id').distinct().count()

print(f'Quantidade de ciclos: {qtd_ciclos}')
print(f'Quantidade total de ciclos na temporada: {qtd_ciclos_total}')
print(f'Quantidade de ciclos em relação ao total: {qtd_ciclos / qtd_ciclos_total:.2%}')

#df_possession_agg.show()

Quantidade de ciclos: 5784
Quantidade total de ciclos na temporada: 100930
Quantidade de ciclos em relação ao total: 5.73%


In [15]:
# (
#     df_cycle_start_gk
#     .filter((F.col('gameId') == 4807) & (F.col('possession_id') == 198))
# ).show(100, truncate=True)

(
    df_cycle_start_gk
    .filter((F.col('gameId') == 4807))
    .select('possession_id', 'period', 'startFormattedGameClock', 'eventTypeDescription', 'eventOutcomeDescription', 'eventPlayerPositionType', 'eventPlayerPositionGroup', 'threat_score',
'is_tracking_inconsistent',
'surface_area',
'stretch_index',
'team_length',
'team_width',
'defense_width',
'height_goal_player',
'height_goal_team_centroide',
'height_goal_def_centroide',
'def_mid_dist',
'def_atk_dist',
'atk_mid_dist',
'numeric_superiority_10m',
'numeric_superiority_20m',
'ball_x')
).show(3000, truncate=True)

+-------------+------+-----------------------+--------------------+-----------------------+-----------------------+------------------------+------------+------------------------+------------+-------------+-----------+----------+-------------+------------------+--------------------------+-------------------------+------------+------------+------------+-----------------------+-----------------------+------+
|possession_id|period|startFormattedGameClock|eventTypeDescription|eventOutcomeDescription|eventPlayerPositionType|eventPlayerPositionGroup|threat_score|is_tracking_inconsistent|surface_area|stretch_index|team_length|team_width|defense_width|height_goal_player|height_goal_team_centroide|height_goal_def_centroide|def_mid_dist|def_atk_dist|atk_mid_dist|numeric_superiority_10m|numeric_superiority_20m|ball_x|
+-------------+------+-----------------------+--------------------+-----------------------+-----------------------+------------------------+------------+------------------------+----

In [16]:
df_possession_agg_pd = df_possession_agg.toPandas()

print(df_possession_agg_pd['qtd_eventos'].describe())

fig = go.Figure()

#cores = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd']

fig.add_trace(
    go.Box(
        y=df_possession_agg_pd['qtd_eventos'],
        name='',
        marker_color='#1f77b4',
        text=df_possession_agg_pd['possession_id'],
        hovertemplate='%{text}<br>Quantidade de eventos: %{y}<extra></extra>',
        #legendgroup=freq,
        #showlegend=True
    )
)

fig.update_layout(
    title=f'Ciclos de posse das partidas',
    height=700,
    width=600
)

fig.update_yaxes(title_text='Quantidade de eventos')
fig.show()

count    5784.000000
mean        4.804115
std         5.523488
min         1.000000
25%         1.000000
50%         3.000000
75%         6.000000
max        64.000000
Name: qtd_eventos, dtype: float64


In [17]:
# Quantos ciclos NAO terminam logo no pico de ameaca — ou seja, o time
# continua com a posse mesmo depois de atingir o threat_score maximo do
# ciclo (o pico nao e o ultimo evento). Usa a mesma logica de F.max_by com
# o mesmo filtro de tracking inconsistente ja usado em build_feature_target_stats_df/
# df_ball_pos_at_max, so que aqui a "coisa escolhida" pelo max_by e a
# posicao (rank) do evento de pico dentro do ciclo, nao a posicao da bola.
w_cycle_order = Window.partitionBy('competitionId', 'season', 'gameId', 'possession_id').orderBy('startGameClock')

df_cycle_ranked = df_cycle_start_gk.withColumn('event_rank', F.row_number().over(w_cycle_order))

consistent_threat_for_peak = F.when(~F.col('is_tracking_inconsistent'), F.col('threat_score'))

df_peak_position = (
    df_cycle_ranked
    .groupBy('competitionId', 'season', 'gameId', 'possession_id')
    .agg(
        F.max_by(F.col('event_rank'), consistent_threat_for_peak).alias('peak_event_rank'),
        F.count('*').alias('total_events'),
    )
    .withColumn('continues_after_peak', F.col('peak_event_rank') < F.col('total_events'))
)

df_peak_position_pd = df_peak_position.toPandas()

qtd_continua = int(df_peak_position_pd['continues_after_peak'].sum())
qtd_total = len(df_peak_position_pd)
print(f'Ciclos que continuam com a posse depois do pico de ameaca: {qtd_continua} / {qtd_total} ({qtd_continua / qtd_total:.2%})')

# Mesma coisa, mas por partida: em cada jogo, qual % dos ciclos "chega no
# maximo e cai" (continua com a posse, threat_score caindo depois do pico,
# em vez de encerrar logo ali)? E qual a MEDIA desse % entre as partidas da
# temporada — diferente do % agregado acima (que pesa mais as partidas com
# mais ciclos), a media por partida da o mesmo peso pra cada jogo.
df_continua_por_jogo = (
    df_peak_position_pd
    .groupby(['competitionId', 'season', 'gameId'])['continues_after_peak']
    .agg(qtd_ciclos='count', qtd_continua='sum')
)
df_continua_por_jogo['pct_continua'] = (df_continua_por_jogo['qtd_continua'] / df_continua_por_jogo['qtd_ciclos']).round(3)

media_pct_continua_temporada = df_continua_por_jogo['pct_continua'].mean()
print(f'Media do % de ciclos que continuam depois do pico, por partida: {media_pct_continua_temporada:.2%}')

#df_continua_por_jogo

Ciclos que continuam com a posse depois do pico de ameaca: 1644 / 5784 (28.42%)
Media do % de ciclos que continuam depois do pico, por partida: 29.55%


In [ ]:
# Mesmo teste, mas por time (defendingTeamName) — mesma logica das analises
# anteriores por time (heatmap por time, faixa por time): confere se esse
# padrao de "chegar no maximo e cair" esta concentrado em times especificos
# ou e uniforme entre eles.
df_cycle_team_keys = (
    df_cycle_start_gk
    .select('competitionId', 'season', 'gameId', 'possession_id', 'defendingTeamName')
    .distinct()
    .toPandas()
)

df_continua_por_time = (
    df_peak_position_pd
    .merge(df_cycle_team_keys, on=['competitionId', 'season', 'gameId', 'possession_id'], how='left')
    .groupby('defendingTeamName')['continues_after_peak']
    .agg(qtd_ciclos='count', qtd_continua='sum')
)
df_continua_por_time['pct_continua'] = (df_continua_por_time['qtd_continua'] / df_continua_por_time['qtd_ciclos']).round(3)

media_pct_continua_por_time = df_continua_por_time['pct_continua'].mean()
print(f'Media do % de ciclos que continuam depois do pico, por time: {media_pct_continua_por_time:.2%}')

df_continua_por_time.sort_values('pct_continua', ascending=False)

In [44]:
features = [
    'surface_area',
    'stretch_index',
    'team_length',
    'team_width',
    'defense_width',
    'height_goal_player',
    'height_goal_team_centroide',
    'height_goal_def_centroide',
    'def_mid_dist',
    'def_atk_dist',
    'atk_mid_dist',
    'numeric_superiority_10m',
    'numeric_superiority_20m',
]
target = 'threat_score'
group_cols = ['competitionId', 'season', 'gameId', 'possession_id']

df_features_target_agg = build_feature_target_stats_df(
    df=df_cycle_start_gk,
    features=features,
    target=target,
    group_cols=group_cols,
)

df_features_target_agg_pd = df_features_target_agg.toPandas()

for feature in features:
    plot_correlation_heatmap(df_features_target_agg_pd, feature, target, target_aggs=['max'])

In [45]:
# mesma análise de cima, mas incluindo o time (defendingTeamName) nas
# chaves de agrupamento — gera um heatmap por time na temporada, além do
# heatmap geral, pra ver se a correlação se sustenta time a time
features = [
    'surface_area',
    'stretch_index',
    'team_length',
    'team_width',
    'defense_width',
    'height_goal_player',
    'height_goal_team_centroide',
    'height_goal_def_centroide',
    'def_mid_dist',
    'def_atk_dist',
    'atk_mid_dist',
    'numeric_superiority_10m',
    'numeric_superiority_20m',
]
target = 'threat_score'
group_cols_by_team = ['competitionId', 'season', 'gameId', 'possession_id', 'defendingTeamName']

df_features_target_agg_by_team = build_feature_target_stats_df(
    df=df_cycle_start_gk,
    features=features,
    target=target,
    group_cols=group_cols_by_team,
)

df_features_target_agg_by_team_pd = df_features_target_agg_by_team.toPandas()

plot_correlation_heatmap_by_team(
    df_features_target_agg_by_team_pd, features, target, team_col='defendingTeamName', feature_agg='min', target_aggs='max', max_teams=5
)

In [ ]:
# sugestão: correlacionar o max_threat_score com as minimas x qtd de gols sofridos por jogo ou xG sofrido

## Estratificação por posição de campo

Complementa o heatmap geral e o heatmap por time: testa se a correlação
entre as features defensivas e o target se sustenta em diferentes zonas do
campo, ou se é puxada por confundimento posicional — o pico de ameaça
tende a acontecer perto do próprio gol, e a compactação defensiva também
aumenta naturalmente ali, então uma correlação alta na base inteira pode
ser só um artefato de "as duas coisas acontecem perto do gol", não um
efeito real da forma defensiva.

In [22]:
# Agregação adicional (separada de build_feature_target_stats_df, mais
# simples que embutir na função genérica): posição x da bola no evento de
# PICO do target dentro do ciclo. Usa F.max_by(ball_x, target) — pega o
# ball_x do evento com o MAIOR target no ciclo, não a posição média do
# ciclo, porque a posição média diluiria justamente o momento que mais
# importa pra essa análise (o pico de ameaça). Sobre df_cycle_start_gk (não
# df_threat_features) — o notebook inteiro é a perspectiva "Abordagem 1"
# (ciclos que começam com GK + 11 defensores), então toda análise de
# features/target aqui embaixo precisa ficar na mesma população. Reaproveita
# "features", "target" e "group_cols" já definidos na análise geral (cima).
# Mesmo filtro de tracking inconsistente usado em build_feature_target_stats_df:
# sem isso, o evento de pico do target (usado pelo max_by) poderia ser
# justamente um evento com tracking inconsistente — daí o ball_x_at_max
# viria de um evento que nem contribuiu pro min/max/avg das features desse
# mesmo ciclo, misturando eventos "válidos" e "descartados" na mesma faixa.
consistent_ball_x = F.when(~F.col('is_tracking_inconsistent'), F.col('ball_x'))
consistent_target_for_max_by = F.when(~F.col('is_tracking_inconsistent'), F.col(target))

df_ball_pos_at_max = (
    df_cycle_start_gk
    .groupBy(*group_cols)
    .agg(F.max_by(consistent_ball_x, consistent_target_for_max_by).alias('ball_x_at_max'))
)

df_features_target_agg_pos = df_features_target_agg.join(df_ball_pos_at_max, on=group_cols, how='inner')

df_features_target_agg_pos_pd = df_features_target_agg_pos.toPandas()

# Divide ball_x_at_max em faixas (bins) DEPOIS da agregação, sobre o pandas
# já pronto — não pode virar chave de groupBy na agregação original porque
# só existe DEPOIS de agregar (é o resultado de um max_by sobre os eventos
# do ciclo); usá-la como chave de entrada seria circular (a agregação
# precisaria do resultado dela mesma pra decidir o agrupamento).
NUM_POSITION_BINS = 4

df_features_target_agg_pos_pd['ball_x_at_max_bin'] = pd.cut(
    df_features_target_agg_pos_pd['ball_x_at_max'], bins=NUM_POSITION_BINS
)

### Teste de composição: as faixas são "zona" ou "posse curta disfarçada"?

Se a faixa com correlação mais forte tiver ciclos sistematicamente mais
curtos (menos eventos) e terminados em perda de bola, o resultado pode ser
efeito de composição (posses abortadas cedo, que nunca saíram de perto do
próprio gol) em vez de comportamento defensivo real naquela zona.

In [23]:
# qtd_eventos por ciclo — reaproveita df_possession_agg_pd (já calculado
# mais acima, agregado sobre df_cycle_start_gk). Antes, df_features_target_agg
# vinha de df_threat_features (todos os ciclos), uma população diferente da
# de df_possession_agg (só os ciclos que começam com GK + 11 defensores) —
# agora que a análise de features/target também usa df_cycle_start_gk, as
# duas populações coincidem e não precisa recalcular nada.
df_composicao_por_faixa = (
    df_features_target_agg_pos_pd
    .merge(
        df_possession_agg_pd,
        on=['competitionId', 'season', 'gameId', 'possession_id'],
        how='left'
    )
    .groupby('ball_x_at_max_bin')['qtd_eventos']
    .agg(['mean', 'median', 'count'])
    .round(2)
)

df_composicao_por_faixa

,mean,median,count
ball_x_at_max_bin,,,
"(-52.603, -24.193]",1.73,1.0,2598
"(-24.193, 4.105]",5.35,4.0,1330
"(4.105, 32.403]",7.25,5.0,969
"(32.403, 60.7]",12.39,10.5,718


In [46]:
plot_correlation_by_position_bin(
    df_features_target_agg_pos_pd, features, target,
    feature_agg='min', target_agg='max', bin_col='ball_x_at_max_bin'
)

In [47]:
# Mesma lógica de ball_x_at_max de cima, mas agrupando também por time
# (group_cols_by_team) — permite repetir a estratificação por zona, agora
# time a time, em vez de misturando todos os times juntos.
consistent_ball_x_by_team = F.when(~F.col('is_tracking_inconsistent'), F.col('ball_x'))
consistent_target_for_max_by_by_team = F.when(~F.col('is_tracking_inconsistent'), F.col(target))

df_ball_pos_at_max_by_team = (
    df_cycle_start_gk
    .groupBy(*group_cols_by_team)
    .agg(F.max_by(consistent_ball_x_by_team, consistent_target_for_max_by_by_team).alias('ball_x_at_max'))
)

df_features_target_agg_by_team_pos = df_features_target_agg_by_team.join(
    df_ball_pos_at_max_by_team, on=group_cols_by_team, how='inner'
)

df_features_target_agg_by_team_pos_pd = df_features_target_agg_by_team_pos.toPandas()

df_features_target_agg_by_team_pos_pd['ball_x_at_max_bin'] = pd.cut(
    df_features_target_agg_by_team_pos_pd['ball_x_at_max'], bins=NUM_POSITION_BINS
)

plot_correlation_by_position_bin_by_team(
    df_features_target_agg_by_team_pos_pd, features, target,
    team_col='defendingTeamName', feature_agg='min', target_agg='max', bin_col='ball_x_at_max_bin', max_teams=10
)

In [48]:
# Teste: mesma analise por faixa, mas só com ciclos com pelo menos 4
# eventos - exclui os ciclos muito curtos (a faixa mais a esquerda tinha
# media de so ~1.7 evento no teste de composicao acima) pra ver se o
# padrao de correlacao por zona se sustenta sem esse "ruido" de posses
# abortadas cedo. Reaproveita os bins ja calculados (ball_x_at_max_bin) -
# so filtra quais ciclos entram em cada correlacao.
MIN_QTD_EVENTOS = 3

df_features_target_agg_pos_min_eventos_pd = (
    df_features_target_agg_pos_pd
    .merge(
        df_possession_agg_pd,
        on=['competitionId', 'season', 'gameId', 'possession_id'],
        how='left'
    )
    .query(f'qtd_eventos >= {MIN_QTD_EVENTOS}')
)

print(df_features_target_agg_pos_min_eventos_pd['ball_x_at_max_bin'].value_counts().sort_index())

plot_correlation_by_position_bin(
    df_features_target_agg_pos_min_eventos_pd, features, target,
    feature_agg='min', target_agg='max', bin_col='ball_x_at_max_bin'
)

ball_x_at_max_bin
(-52.603, -24.193]     483
(-24.193, 4.105]      1033
(4.105, 32.403]        853
(32.403, 60.7]         704
Name: count, dtype: int64


In [49]:
# Correlacoes por time nessa mesma reducao de ciclo (qtd_eventos >=
# MIN_QTD_EVENTOS) - reaproveita df_features_target_agg_by_team_pos_pd
# (features/target/bin por time, ja calculado acima) e so filtra pelos
# ciclos com eventos suficientes antes de plotar, igual ao teste sem time
# feito mais acima.
df_features_target_agg_by_team_pos_min_eventos_pd = (
    df_features_target_agg_by_team_pos_pd
    .merge(df_possession_agg_pd, on=['competitionId', 'season', 'gameId', 'possession_id'], how='left')
    .query(f'qtd_eventos >= {MIN_QTD_EVENTOS}')
)

plot_correlation_by_position_bin_by_team(
    df_features_target_agg_by_team_pos_min_eventos_pd, features, target,
    team_col='defendingTeamName', feature_agg='min', target_agg='max', bin_col='ball_x_at_max_bin', max_teams=10
)

In [ ]:
# testar todas as correlações trazendo um * ou coisa do tipo indicando que tem significancia estatistica

# pegar variaveis agregadas mais correlacionadas de forma absoluta la no inicio e testar nas correlações seguintes 

# levar essas melhores agregações pra uma v0 do modelo modelo (dps daria pra explorar todas as variações de uma vez pro modelo e ver pela explicabilidade qual foi melhor e se bate com a eda)

# começar modelo com arvore de decisão e regressão linear

# ver resultados/performance e pensar no que da pra melhorar (features, algoritmos, etc)



- OBS: A intenção é trazer uma v0 pra qualificação, mas é possivel melhorar o modelo pensando em estratégias pós qualificação para dissertação ou artigos

- OBS2: A analise olhando ciclos começando no goleiro e os 11 atrás foi melhor que a começando com apenas 11 atrás, pois sugere-se que dá pra ver melhor as estratificações em campo, enquanto que sem começar no goleiro pode começar em qualquer parte do campo e fica pior.

Os times mostraram ter variações muito signficativas, mas o modelo deve ser um formato default primeiro como v0 e considerando todos dentro do mesmo bolo.

Legal que cortar eventos curtos reduziu ruido no primeiro quadrante à esquerda e mostrou caracteristicas de sucesso na performance a respeito de pressão defensiva alta entre os times. O problema é que variando o limiar minimo de evento pro ciclo ser valido reduz a amosra que já é pequena. Vale testar mudando esse limiar caso necessário, mas a v0 ser sem esse corte ou começando com u mlimiar baixo de 1-2 eventos.